# DepthNet + Servo Decision Demo

This notebook runs a small fusion test on the JETANK:

1. Capture a frame with JetBot `Camera.instance()`.
2. Run `jetson-inference` DepthNet on that frame.
3. Print center-depth statistics so you can see changes when you place an object in front of the camera.
4. Use the tutorial's simple `TTLServo.servoAngleCtrl()` function to make one of two tiny visible servo motions.

Run cells one by one. Avoid **Run All** while testing hardware.

## 1. Configuration

`videoSource(csi://0)` did not return frames on this robot, so this notebook uses JetBot Camera and converts the NumPy frame with `cudaFromNumpy()`.

The threshold is intentionally simple: if the center depth mean is below `DEPTH_THRESHOLD`, the camera/arm pan servo moves left-right; otherwise the tilt servo moves up-down. The depth value is relative, not calibrated distance in meters.

In [ ]:
JETSON_INFERENCE_ROOT = "/workspace/jetson-inference"

WIDTH = 320
HEIGHT = 240
NETWORK = "fcn-mobilenet"
RUN_SECONDS = 20.0
WARMUP_FRAMES = 2
LOOP_PAUSE_SECONDS = 1.0

ENABLE_ARM_SIGNAL = True
DEPTH_THRESHOLD = 1.6
PAN_SERVO_ID = 1
TILT_SERVO_ID = 5
PAN_ANGLE = 10
TILT_ANGLE = 8
HOME_ANGLE = 0
SERVO_SPEED = 180
SERVO_SETTLE_SECONDS = 0.18

## 2. Imports and Path Setup

The `jetson_inference` and `jetson_utils` Python bindings are loaded from the existing `/workspace/jetson-inference` build on the robot.

In [ ]:
import os
import sys
import time

extra_paths = [
    f"{JETSON_INFERENCE_ROOT}/build/aarch64/lib/python/3.6",
    f"{JETSON_INFERENCE_ROOT}/python/examples",
]

for path in extra_paths:
    if path not in sys.path:
        sys.path.insert(0, path)

lib_path = f"{JETSON_INFERENCE_ROOT}/build/aarch64/lib"
os.environ["LD_LIBRARY_PATH"] = lib_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

import cv2
import numpy as np

from jetbot import Camera
from jetson_inference import depthNet
from jetson_utils import cudaDeviceSynchronize, cudaFromNumpy, cudaToNumpy

print("imports ok")

## 3. Servo Helpers

This section imports `TTLServo` and defines two small signal motions:

- **Pan motion** on servo 1: left-right-center.
- **Tilt motion** on servo 5: up-down-center.

Both motions use small angles and return to `0` so the robot is easy to reset between frames.

In [ ]:
TTLServo = None

if ENABLE_ARM_SIGNAL:
    try:
        from SCSCtrl import TTLServo
        print("arm signal enabled")
    except ImportError as exc:
        print(f"arm signal disabled; SCSCtrl import failed: {exc}")
else:
    print("arm signal disabled by config")


def servo_swing(servo_id, angle, label):
    """Move one servo both ways, then return it to center."""
    if TTLServo is None:
        print(f"[arm] {label} skipped")
        return

    TTLServo.servoAngleCtrl(servo_id, angle, 1, SERVO_SPEED)
    time.sleep(SERVO_SETTLE_SECONDS)
    TTLServo.servoAngleCtrl(servo_id, -angle, 1, SERVO_SPEED)
    time.sleep(SERVO_SETTLE_SECONDS)
    TTLServo.servoAngleCtrl(servo_id, HOME_ANGLE, 1, SERVO_SPEED)
    time.sleep(SERVO_SETTLE_SECONDS)
    print(f"[arm] {label} swing complete")


def depth_signal_motion(mean_depth):
    """Choose one of two visible servo motions from the center depth estimate."""
    if mean_depth < DEPTH_THRESHOLD:
        print(f"[decision] mean_depth={mean_depth:.3f} < {DEPTH_THRESHOLD}; pan left/right")
        servo_swing(PAN_SERVO_ID, PAN_ANGLE, "pan")
    else:
        print(f"[decision] mean_depth={mean_depth:.3f} >= {DEPTH_THRESHOLD}; tilt up/down")
        servo_swing(TILT_SERVO_ID, TILT_ANGLE, "tilt")

## 4. Load DepthNet and Camera

This loads the TensorRT DepthNet model and starts the JetBot camera. The first load may take a little while, but the model engine should already be cached on the robot.

In [ ]:
print("[depth] loading network...")
net = depthNet(NETWORK)
depth_field = net.GetDepthField()
depth_numpy = cudaToNumpy(depth_field)

print(f"[depth] network={net.GetNetworkName()}")
print(f"[depth] field={depth_field.width}x{depth_field.height}, format={depth_field.format}")

print(f"[camera] starting JetBot Camera {WIDTH}x{HEIGHT}")
camera = Camera.instance(width=WIDTH, height=HEIGHT)
time.sleep(1.0)

frame = camera.value
print(type(frame), None if frame is None else frame.shape, None if frame is None else frame.dtype)

## 5. Warmup

DepthNet and CUDA can be slow on the first frame. These warmup frames are processed before the real timed loop starts, so the demo does not end after one expensive first frame.

In [ ]:
print(f"[warmup] processing {WARMUP_FRAMES} frame(s) before starting timer")
warmup_done = 0
while warmup_done < WARMUP_FRAMES:
    frame = camera.value
    if frame is None:
        print("[warmup] no camera frame")
        time.sleep(0.2)
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    cuda_img = cudaFromNumpy(rgb)
    net.Process(cuda_img)
    cudaDeviceSynchronize()
    warmup_done += 1
    print(f"[warmup] frame {warmup_done}/{WARMUP_FRAMES} processed")
    time.sleep(0.1)

## 6. Run Timed Decision Loop

During this cell, try changing the scene in front of the camera, for example by placing a water bottle near the center of the frame. Watch the printed `mean` and `delta`, and watch which servo motion is selected.

In [ ]:
def summarize_center_depth(depth_array):
    """Return mean/min/max from the center 20% of the raw depth field."""
    h, w = depth_array.shape[:2]
    x1, x2 = int(w * 0.4), int(w * 0.6)
    y1, y2 = int(h * 0.4), int(h * 0.6)
    center = depth_array[y1:y2, x1:x2]
    return float(np.mean(center)), float(np.min(center)), float(np.max(center))


successful_frames = 0
start_time = time.time()
previous_mean_depth = None

try:
    while time.time() - start_time < RUN_SECONDS:
        frame = camera.value
        if frame is None:
            print("[camera] no frame")
            time.sleep(0.2)
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        cuda_img = cudaFromNumpy(rgb)

        net.Process(cuda_img)
        cudaDeviceSynchronize()

        successful_frames += 1
        mean_depth, min_depth, max_depth = summarize_center_depth(depth_numpy)
        elapsed = time.time() - start_time

        if previous_mean_depth is None:
            delta_text = "delta=n/a"
        else:
            delta_text = f"delta={mean_depth - previous_mean_depth:+.3f}"

        print(
            f"[{elapsed:05.2f}s frame={successful_frames:03d}] "
            f"center_depth mean={mean_depth:.3f}, min={min_depth:.3f}, "
            f"max={max_depth:.3f}, {delta_text}"
        )

        depth_signal_motion(mean_depth)
        previous_mean_depth = mean_depth
        time.sleep(LOOP_PAUSE_SECONDS)
finally:
    print(f"done; successful_frames={successful_frames}")

## 7. Cleanup

Run this before re-running camera setup, or when you are done testing. On Jetson/Argus, cleanup may still print warnings; the important part is that the camera is released for the next run.

In [ ]:
try:
    camera.stop()
    time.sleep(1.0)
    print("camera stopped")
except Exception as exc:
    print(f"camera stop failed: {exc}")